In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree
import joblib
import folium
import warnings
warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
SESSIONS_CSV = r"C:\\Users\\Yash\\OneDrive\\Desktop\\Yash\\Evision\\Evision\\Files\\Code Generated CSV\\initial_sessions_master.csv"
STATIONS_CSV = r"C:\\Users\\Yash\\OneDrive\\Desktop\\Yash\\Evision\\Evision\\Files\\Code Generated CSV\\initial_station_aggregates.csv"

OUTPUT_MODEL = "ridge_model.pkl"
OUTPUT_SUGGESTIONS = "ridge_regression_suggested_sites.csv"
OUTPUT_MAP = "ridge_regression_suggested_sites_map.html"
OUTPUT_MODEL_COMP = "ridge_model_results.csv"

MIN_DISTANCE_FOR_FAR_KM = 2.0
MAX_NEAR_DISTANCE_KM = 0.5
TOP_N_PER_BUCKET = 20
NEARBY_RADIUS_KM = 2.0
RANDOM_STATE = 42
N_FOLDS = 5

# ---------- HELPERS ----------
def find_latlon_columns(df):
    lat_candidates = ["latitude","lat","Latitude","LAT"]
    lon_candidates = ["longitude","lon","Longitude","LON","lng","Lng"]
    for lat in lat_candidates:
        for lon in lon_candidates:
            if lat in df.columns and lon in df.columns:
                return lat, lon
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric) >= 2:
        return numeric[0], numeric[1]
    raise ValueError("No lat/lon columns found.")

def haversine_balltree_min_dist_km(points_deg, existing_deg):
    if existing_deg.shape[0] == 0:
        return np.full(points_deg.shape[0], np.inf)
    earth_r = 6371.0
    tree = BallTree(np.radians(existing_deg), metric='haversine')
    dist_rad, _ = tree.query(np.radians(points_deg), k=1)
    return (dist_rad.flatten() * earth_r)

def haversine_counts_within(points_deg, other_deg, radius_km):
    if other_deg.shape[0] == 0:
        return np.zeros(points_deg.shape[0], dtype=int)
    earth_r = 6371.0
    tree = BallTree(np.radians(other_deg), metric='haversine')
    rad = radius_km / earth_r
    ind = tree.query_radius(np.radians(points_deg), r=rad)
    return np.array([len(a) for a in ind])

# ---------- 1) Load data ----------
if not os.path.exists(SESSIONS_CSV) or not os.path.exists(STATIONS_CSV):
    raise FileNotFoundError("Make sure both CSV paths are correct.")

sessions = pd.read_csv(SESSIONS_CSV)
stations = pd.read_csv(STATIONS_CSV)

s_lat, s_lon = find_latlon_columns(sessions)
e_lat, e_lon = find_latlon_columns(stations)
print(f"Session coords: {s_lat},{s_lon} | Station coords: {e_lat},{e_lon}")
print(f"Rows -> sessions: {len(sessions)}, stations: {len(stations)}")

# ---------- 2) Choose target column ----------
possible_targets = [
    "avg_sessions_per_month",
    "total_sessions",
    "total_energy_kWh",
    "avg_energy_per_session_kWh",
    "sessions_count",
    "sessions", "session_count"
]
target_col = None
for c in possible_targets:
    if c in stations.columns:
        target_col = c
        break
if target_col is None:
    for c in stations.select_dtypes(include=[np.number]).columns:
        if c not in [e_lat, e_lon]:
            target_col = c
            break
if target_col is None:
    raise ValueError("No target column found in station CSV.")
print("Using target column:", target_col)

# ---------- 3) Build features ----------
session_pts = sessions[[s_lat, s_lon]].dropna().to_numpy()
station_pts = stations[[e_lat, e_lon]].dropna().to_numpy()

stations['nearby_sessions_2km'] = haversine_counts_within(station_pts, session_pts, NEARBY_RADIUS_KM) if session_pts.size and station_pts.size else 0
stations['competitors_within_2km'] = haversine_counts_within(station_pts, station_pts, NEARBY_RADIUS_KM) - 1 if station_pts.size and station_pts.size else 0

if station_pts.shape[0] > 1:
    tree = BallTree(np.radians(station_pts), metric='haversine')
    dist_rad, ind = tree.query(np.radians(station_pts), k=2)
    stations['distance_to_nearest_station_km'] = dist_rad[:,1] * 6371.0
else:
    stations['distance_to_nearest_station_km'] = np.inf

numeric_cols = stations.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [c for c in numeric_cols if c != target_col]
if len(feature_columns) == 0:
    stations['nearby_sessions_2km'] = 0
    feature_columns = ['nearby_sessions_2km']

print("Feature columns:", feature_columns)

train = stations[feature_columns + [target_col]].dropna(subset=[target_col]).copy()
X = train[feature_columns].fillna(0)
y = train[target_col].values
print("Training rows:", len(train))

# ---------- 4) Setup Ridge model ----------
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(random_state=RANDOM_STATE))
])

# ---------- 5) Cross-validation ----------
cv = KFold(n_splits=min(N_FOLDS, max(2, len(X)//2)), shuffle=True, random_state=RANDOM_STATE)
scores_rmse = -cross_val_score(ridge_model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
scores_r2 = cross_val_score(ridge_model, X, y, cv=cv, scoring="r2")

results_df = pd.DataFrame([{
    "model": "Ridge",
    "rmse_mean": float(scores_rmse.mean()),
    "rmse_std": float(scores_rmse.std()),
    "r2_mean": float(scores_r2.mean()),
    "r2_std": float(scores_r2.std())
}])
results_df.to_csv(OUTPUT_MODEL_COMP, index=False)

print(f"Ridge Model CV -> RMSE={scores_rmse.mean():.4f} (+/-{scores_rmse.std():.4f}), R²={scores_r2.mean():.4f}")

# ---------- 6) Fit on full data ----------
ridge_model.fit(X, y)
joblib.dump(ridge_model, OUTPUT_MODEL)
print("Saved Ridge model to", OUTPUT_MODEL)

# ---------- 7) Generate candidate sites ----------
sessions['lat_round'] = sessions[s_lat].round(4)
sessions['lon_round'] = sessions[s_lon].round(4)
candidates = sessions.groupby(['lat_round','lon_round']).size().reset_index(name='session_count')
cand_pts = candidates[['lat_round','lon_round']].to_numpy()
exist_pts = stations[[e_lat, e_lon]].to_numpy()

candidates['min_dist_km'] = haversine_balltree_min_dist_km(cand_pts, exist_pts) if exist_pts.size else np.inf
candidates['nearby_sessions_2km'] = haversine_counts_within(cand_pts, session_pts, NEARBY_RADIUS_KM) if session_pts.size else 0
candidates['competitors_within_2km'] = haversine_counts_within(cand_pts, exist_pts, NEARBY_RADIUS_KM) if exist_pts.size else 0

cand_feat = pd.DataFrame(0, index=candidates.index, columns=feature_columns)
for col in ['nearby_sessions_2km','competitors_within_2km','min_dist_km','distance_to_nearest_station_km']:
    if col in candidates.columns:
        cand_feat[col] = candidates[col].values
    else:
        cand_feat[col] = 0

try:
    if exist_pts.size:
        tree = BallTree(np.radians(exist_pts), metric='haversine')
        _, idx = tree.query(np.radians(cand_pts), k=1)
        idx = idx.flatten()
        station_numeric = stations[feature_columns].reset_index(drop=True)
        mapped = station_numeric.iloc[idx].reset_index(drop=True)
        for c in feature_columns:
            if c in mapped.columns and c not in ['nearby_sessions_2km','competitors_within_2km','distance_to_nearest_station_km','min_dist_km']:
                cand_feat[c] = mapped[c].values
except Exception:
    pass

cand_feat = cand_feat.fillna(0)[feature_columns]
candidates['predicted_demand'] = ridge_model.predict(cand_feat)

near_bucket = candidates[candidates['min_dist_km'] <= MAX_NEAR_DISTANCE_KM].sort_values('predicted_demand', ascending=False).head(TOP_N_PER_BUCKET).assign(bucket='near')
far_bucket = candidates[candidates['min_dist_km'] >= MIN_DISTANCE_FOR_FAR_KM].sort_values('predicted_demand', ascending=False).head(TOP_N_PER_BUCKET).assign(bucket='far')
combined = pd.concat([near_bucket, far_bucket]).reset_index(drop=True)
combined.to_csv(OUTPUT_SUGGESTIONS, index=False)
print("Saved candidate sites to", OUTPUT_SUGGESTIONS)

# ---------- 8) Save map ----------
center = [sessions[s_lat].mean(), sessions[s_lon].mean()]
m = folium.Map(location=center, zoom_start=11)

for _, r in stations.iterrows():
    folium.Marker(
        location=[r[e_lat], r[e_lon]],
        icon=folium.Icon(color="green", icon="ok-sign"),
        popup=f"Existing Station: {r.get('station_name','Unknown')} | {target_col}: {r.get(target_col)}"
    ).add_to(m)

maxpred = combined['predicted_demand'].max() if not combined.empty else 1.0
for _, r in combined.iterrows():
    folium.Marker(
        location=[r['lat_round'], r['lon_round']],
        icon=folium.Icon(color="red", icon="flag"),
        popup=f"Suggested New Station | Predicted demand: {r['predicted_demand']:.2f} | "
              f"Bucket: {r['bucket']} | Dist from nearest: {r['min_dist_km']:.2f} km"
    ).add_to(m)

m.save(OUTPUT_MAP)
print("Saved map to", OUTPUT_MAP)

# ---------- 9) Summary ----------
print("\nFinal results:")
print(results_df.head())
print(f"Ridge Model | CV RMSE: {scores_rmse.mean():.4f} | CV R²: {scores_r2.mean():.4f} ({scores_r2.mean()*100:.2f}%)")


Session coords: latitude,longitude | Station coords: latitude,longitude
Rows -> sessions: 262, stations: 154
Using target column: sessions_count
Feature columns: ['energy_consumed_kwh_sum', 'energy_consumed_kwh_mean', 'charging_duration_hours_mean', 'charging_rate_kw_mean', 'latitude', 'longitude', 'stations_within_2km', 'nearby_sessions_2km', 'competitors_within_2km', 'distance_to_nearest_station_km']
Training rows: 154
Ridge Model CV -> RMSE=0.7249 (+/-0.5638), R²=0.5805
Saved Ridge model to ridge_model.pkl
Saved candidate sites to ridge_regression_suggested_sites.csv
Saved map to ridge_regression_suggested_sites_map.html

Final results:
   model  rmse_mean  rmse_std   r2_mean    r2_std
0  Ridge   0.724863  0.563836  0.580548  0.364485
Ridge Model | CV RMSE: 0.7249 | CV R²: 0.5805 (58.05%)
